bsklearn01_linear_model 逻辑回归，通常用于离散值，用于分类

In [ ]:
# 心脏病预测 heart-disease
# 特征信息
# Age: 患者年龄 [岁]
# Sex: 患者性别 [M: 男性, F: 女性]
# ChestPainType: 胸痛类型 [TA: 典型心绞痛, ATA: 非典型心绞痛, NAP: 非心绞痛, ASY: 无症状]
# RestingBP: 静息血压 [mm Hg]
# Cholesterol: 血清胆固醇 [mm/dl]
# FastingBS: 空腹血糖 [1: 如果空腹血糖 > 120 mg/dl, 0: 否则]
# RestingECG: 静息心电图结果 [Normal: 正常, ST: ST-T波异常 (T波倒置和/或ST段抬高或降低 > 0.05 mV), LVH: 按照Estes标准显示可能或明确的左心室肥大]
# MaxHR: 最大心率 [60 到 202 之间的数值]
# ExerciseAngina: 运动诱发的心绞痛 [Y: 有, N: 无]
# Oldpeak: Oldpeak = ST [测量的ST段抑制的数值]
# ST_Slope: 峰值运动ST段的斜率 [Up: 上升, Flat: 平坦, Down: 下降]
# HeartDisease: 输出类别 [1: 心脏病, 0: 正常]


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

In [6]:
# 阶段 1：加载数据集
# 使用 Pandas 读取 CSV 文件，查看前几行、数据量、列名和数据类型。

data = pd.read_csv('../../../data/ml/heart-disease.csv')
print('Shape of the data is ', data.shape)

data.head()

Shape of the data is  (1025, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


In [10]:
# 阶段 2：数据探索与清洗
# 检查缺失值、重复行、异常值、类别分布以及目标列的含义。

print(data.describe())

count = data.isna().any(axis=1).sum()
print(f"There are {count} missing values.")

               age          sex           cp     trestbps        chol  \
count  1025.000000  1025.000000  1025.000000  1025.000000  1025.00000   
mean     54.434146     0.695610     0.942439   131.611707   246.00000   
std       9.072290     0.460373     1.029641    17.516718    51.59251   
min      29.000000     0.000000     0.000000    94.000000   126.00000   
25%      48.000000     0.000000     0.000000   120.000000   211.00000   
50%      56.000000     1.000000     1.000000   130.000000   240.00000   
75%      61.000000     1.000000     2.000000   140.000000   275.00000   
max      77.000000     1.000000     3.000000   200.000000   564.00000   

               fbs      restecg      thalach        exang      oldpeak  \
count  1025.000000  1025.000000  1025.000000  1025.000000  1025.000000   
mean      0.149268     0.529756   149.114146     0.336585     1.071512   
std       0.356527     0.527878    23.005724     0.472772     1.175053   
min       0.000000     0.000000    71.000000  

In [18]:
# 阶段 3：定义特征 X 和目标 y
# X 是患者特征，y 是心脏病标签，通常为 0/1。

target_columns = "target"
X = data.drop(target_columns, axis=1, inplace=False)
y = data[target_columns]

# X = X.to_numpy()
# y = y.to_numpy()
print(len(X), len(y))
print(X.shape, y.shape)


1025 1025
(1025, 13) (1025,)


In [25]:
# 阶段 5：特征预处理
# 处理缺失值、类别特征，并对数值特征进行标准化。
def norm(input: pd.array) -> pd.array:
    """ 最大最小标准化数据 """
    min = input.min()
    max = input.max()
    if min == max:
        return np.zeros_like(input)
    return (input - min) / (max - min)
# norm_x = pd.array([1,2,3,4,5])
# print(norm(norm_x))

X["age"] = norm(X["age"])
X["trestbps"] = norm(X["trestbps"])
X["chol"] = norm(X["chol"])
X["thalach"] = norm(X["thalach"])
print(X["age"][0:10])


0    0.479167
1    0.500000
2    0.854167
3    0.666667
4    0.687500
5    0.604167
6    0.604167
7    0.541667
8    0.354167
9    0.520833
Name: age, dtype: float64


In [19]:
# 阶段 4：划分训练集、验证集和测试集
# 训练集拟合模型，验证集选择模型与阈值，测试集用于最终评估。

from sklearn.model_selection import train_test_split

# train:validation:test = 8:1:1
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.2, random_state=23)
X_validation, X_test, y_validation, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=23)

print(X_train.shape, X_validation.shape, X_test.shape, y_train.shape, y_validation.shape, y_test.shape)

(820, 13) (102, 13) (103, 13) (820,) (102,) (103,)


In [45]:
# 阶段 6：训练逻辑回归模型
# 使用训练集拟合 Logistic Regression，学习特征权重和偏置。
model = LogisticRegression(
    fit_intercept=True,
    C=1.0,
    solver="lbfgs",
    max_iter=1000,
    random_state=23
)
model.fit(X_train, y_train)

print(model.coef_)
print(model.intercept_)


# val = model.predict(X_validation)
# val_proba = model.predict_proba(X_validation)[:, 1]
# print(val[0:10],y_validation[0:10].to_numpy(),val[0:10] == y_validation[0:10].to_numpy(), val_proba[0:10])

[[-0.00896114 -1.83597232  0.74055501 -0.01663104 -0.00529934  0.10779271
   0.2857879   0.02138947 -0.9524453  -0.58429356  0.50459834 -0.71713031
  -0.89567974]]
[3.98462114]
[0 1 1 1 1 1 1 0 0 0] [0 1 1 1 1 1 1 0 0 0] [ True  True  True  True  True  True  True  True  True  True] [0.03919316 0.84636343 0.8848392  0.99069868 0.65448709 0.97418814
 0.92542169 0.15056777 0.00632523 0.01910743]


In [38]:
# 阶段 7：验证与模型选择
# 使用验证集比较参数、阈值和指标，确定最终模型。
pred_y = model.predict(X_validation)
flag = pred_y==y_validation
count = flag.sum()

print(f"X_validation size: {len(X_validation)}, pred correct count: {count}, correct rate: {count / len(pred_y):.3f}")

X_validation size: 102, pred correct count: 87, correct rate: 0.853


In [47]:
# 阶段 8：测试集评估
# 计算分类指标，分析误判情况，评估模型的泛化能力。
from sklearn.metrics import (
    log_loss,
    accuracy_score,
    zero_one_loss,
    confusion_matrix,
    classification_report,
    roc_auc_score,
)

# 4. 计算训练集和验证集的预测概率
train_proba = model.predict_proba(X_train)[:, 1]
val_proba = model.predict_proba(X_validation)[:, 1]

# 使用默认阈值 0.5 生成预测类别
val_pred = model.predict(X_validation)

# 6. 计算验证集分类误差和其他指标
accuracy = accuracy_score(y_validation, val_pred)
error_rate = zero_one_loss(y_validation, val_pred)

print("\n验证集分类指标：")
print(f"Accuracy（准确率）：{accuracy:.4f}")
print(f"Error Rate（分类误差率）：{error_rate:.4f}")
print(f"Precision、Recall、F1：")
print(classification_report(y_validation, val_pred, zero_division=0))

print("混淆矩阵：")
print(confusion_matrix(y_validation, val_pred, labels=[0, 1]))

# ROC-AUC 使用预测概率计算
if pd.Series(y_validation).nunique() == 2:
    auc = roc_auc_score(y_validation, val_proba)
    print(f"ROC-AUC：{auc:.4f}")
else:
    print("验证集只包含一个类别，无法计算 ROC-AUC")


验证集分类指标：
Accuracy（准确率）：0.8529
Error Rate（分类误差率）：0.1471
Precision、Recall、F1：
              precision    recall  f1-score   support

           0       0.89      0.80      0.84        50
           1       0.82      0.90      0.86        52

    accuracy                           0.85       102
   macro avg       0.86      0.85      0.85       102
weighted avg       0.86      0.85      0.85       102

混淆矩阵：
[[40 10]
 [ 5 47]]
ROC-AUC：0.9173


In [ ]:


# 阶段 9：保存与预测
#
# 保存完整预处理与模型流程，用于新样本预测。
#
